# 08. Voice Agent — STT/Agent/TTS Pipeline

We build a voice agent that converts voice input to text (STT), processes it by the LangChain agent, and then synthesizes text into speech (TTS) and returns it in real-time.

This architecture is a **Sandwich pattern** (STT -> Agent -> TTS), where each layer is connected by streaming, targeting sub-700ms latency.

### Core design principles

The core of the Sandwich architecture is **Streaming Chaining**. Rather than waiting for the complete output of the previous layer, each layer passes partial results to the next layer as soon as they are produced:

- **STT**: Generates partial transcripts in real-time and emits final transcripts when detecting end-of-speech.
- **Agent**: Streams the response token by token through `astream()` — does not wait for the entire response generation to complete.
- **TTS**: Start audio synthesis as soon as a text chunk arrives based on WebSocket

Thanks to this structure, the latency of the entire pipeline is reduced to **the sum of the time until the first output of each stage**, rather than the sum of each stage.

## Learning Objectives

After completing this notebook, you will be able to:

1. **Sandwich Architecture** — Can explain the structure and data flow of STT -> Agent -> TTS pipeline
2. **Architecture Comparison** — You can compare the pros and cons of the Sandwich method and the Speech-to-Speech (S2S) method.
3. **STT Step** — You can understand the Producer-Consumer pattern of AssemblyAI real-time transcription.
4. **Agent Phase** — You can build an agent that generates streaming responses with LangChain `create_agent`
5. **TTS Step** — Understand the operating principle of Cartesia WebSocket-based streaming voice synthesis.
6. **RunnableGenerator** — Pipelines can be combined with asynchronous generator chaining.
7. **Performance Optimization** — Understand optimization techniques at each stage to achieve latency goals.

> **Note**: STT (AssemblyAI) and TTS (Cartesia) are external paid services. The cells are provided as markdown to illustrate the concept, **only the agent creation part is the actual executable code**.

## 8.1 Environment Setup

These are the packages and each role required for a voice agent:

| package | Role |
|--------|------|
| `langchain`, `langchain-openai` | Create Agent and Connect LLM |
| `assemblyai` | Real-time speech-to-text (STT) |
| `cartesia` | Low-latency text-to-speech synthesis (TTS) |
| `websockets` | Real-time two-way communication server |
| `pyaudio` | Microphone audio capture (optional) |

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 8.2 Voice Agent Architecture Overview

The voice agent consists of a 3-layer pipeline:

```
Audio In -> [STT: AssemblyAI] -> Text -> [Agent: LangChain] -> Text -> [TTS: Cartesia] -> Audio Out
```

| 레이어 | 제공자 | 역할 |
|--------|--------|------|
| **STT** | AssemblyAI | 사용자 음성을 텍스트로 변환 |
| **Agent** | LangChain | 텍스트 쿼리를 처리하고 응답 생성 |
| **TTS** | Cartesia | 에이전트의 텍스트 응답을 음성으로 변환 |

각 레이어가 **스트리밍**으로 연결되어, 이전 단계의 완전한 출력을 기다리지 않고 부분 결과를 즉시 __T10__로 전달합니다:

1. **STT** — 부분 전사 결과를 스트리밍 (발화 완료 감지 시 최종 전사)
2. **Agent** — 토큰 단위 스트리밍 응답 생성 (`astream()`)
3. **TTS** — Start speech synthesis before the entire response is complete (WebSocket-based)

### Why Streaming Matters

When processing synchronously without streaming, Next Steps does not start until each step is completely finished. For example, if the agent takes 3 seconds to generate a 200 token response, TTS will only start after 3 seconds. In streaming, on the other hand, TTS begins synthesizing speech **as soon as the agent's first token appears**, so the user already starts hearing speech while the agent is still generating a response.

## 8.3 Architecture comparison table

Compare two approaches to building voice agents.

### Sandwich architecture (STT -> Agent -> TTS)

| Item | Content |
|------|------|
| **Advantages** | Each component can be replaced independently, easy to use text-based tool, convenient for debugging (intermediate text logging) |
| **Disadvantages** | Latency accumulation and loss of non-verbal information such as emotion/intonation due to 3-step serial processing |
| **Suitable for** | tool calling Complex agents with many, when multilingual support is required |

### Speech-to-Speech (S2S)

| Item | Content |
|------|------|
| **Advantages** | Low latency, preservation of voice characteristics (emotion, stress), natural conversation |
| **Disadvantages** | tool calling Difficulty in integration, limited model selection, difficulty in debugging |
| **Suitable for** | Simple conversational interface, when emotion recognition is important |

> In this notebook, we focus on the **tool calling-enabled Sandwich architecture**.

## 8.4 STT Step -- AssemblyAI Real-Time Transcription

> **This cell requires an AssemblyAI API key.** Reference code to understand the concept.

AssemblyAI's `RealtimeTranscriber` operates with the **Producer-Consumer pattern**:

- **Producer**: Send audio chunks captured from the microphone to WebSocket
- **Consumer**: Receives partial and final transcription results as a callback.

Since both operations take place **simultaneously**, you can receive a transcription of the previous utterance even while your voice is being transmitted.

### Two types of transcription results

| Type | class | Description |
|------|--------|------|
| **Partial** | `RealtimeTranscript` | Temporary transcription of words still being uttered. Updated in real-time as the user speaks |
| **Final** | `RealtimeFinalTranscript` | Confirmation transcription after the end of utterance (endpointing) is detected. Pass only this result to the agent |

AssemblyAI's built-in Voice Activity Detection (VAD) automatically detects when an utterance has ended, eliminating the need for separate silence detection logic.

```python
import assemblyai as aai

aai.settings.api_key = "your-assemblyai-key"

transcriber = aai.RealtimeTranscriber(
    sample_rate=16000,
    encoding=aai.AudioEncoding.pcm_s16le,
    on_data=on_transcription_data,
    on_error=on_transcription_error,
)

def on_transcription_data(transcript: aai.RealtimeTranscript):
    if isinstance(transcript, aai.RealtimeFinalTranscript):
        process_user_input(transcript.text)

def on_transcription_error(error: aai.RealtimeError):
    print(f"Transcription error: {error}")

transcriber.connect()
```

### 마이크 오디오 캡처

PCM 16-bit, 16kHz 단일 채널 오디오를 캡처하여 실시간으로 전사기에 전송합니다:

```python
import pyaudio

audio = pyaudio.PyAudio()
stream = audio.open(
    format=pyaudio.paInt16,
    channels=1, rate=16000,
    input=True, frames_per_buffer=1024,
)

while True:
    data = stream.read(1024)
    transcriber.stream(data)
```

## 8.5 Agent Phase -- Utilizing LangChain Agent

The agent stage is the core of the pipeline. An agent created with `create_agent` takes text input, performs inference with tool calling, and streams a text response.

The key to system prompts for voice agents is to elicit concise, conversational responses. For voice output, unlike text, long responses can be burdensome to the user, so users are instructed to generate short responses of no more than 1 to 2 sentences.

### The role of asynchronous streaming

The agent's `astream()` method yields a response token as soon as it is generated. Why this is especially important for voice agents:

- **TTS Early Start**: Speech synthesis is possible from the first token without waiting for the entire response to be completed.
- **Reduced perceived lag**: Users start hearing the agent’s response sooner.
- **Pipeline efficiency**: Agent and TTS run simultaneously, reducing total processing time

In [ ]:
from langchain.agents import create_agent

def search_tool(query: str) -> str:
    """Search the web for the latest information."""
    return f"검색 결과: {query}"

def calendar_tool(action: str, details: str) -> str:
    """Manage calendar events."""
    return f"캘린더 {action}: {details}"

In [ ]:
agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, calendar_tool],
    system_prompt=(
        "You are a useful voice assistant."
        "Keep your responses brief and conversational."
    ),
)

### Generate asynchronous streaming response

`astream()` yields the agent's response token as soon as it is generated. Allows the TTS stage to start speech synthesis without waiting for the entire response to complete.

LangChain's streaming offers two main methods:

| method | Description |
|--------|------|
| `astream()` | Stream the output for each step of agent execution. Contains messages, tool calling results, etc. |
| `astream_events()` | More granular event-level streaming. Provide detailed events such as individual tokens, LLM start/end, etc. |

The voice agent receives message chunks as `astream()`, extracts `content` from each chunk, and passes it to TTS.

In [ ]:
async def stream_agent_response(user_text: str):
    """Stream agent response tokens one by one."""
    async for chunk in agent.astream(
        {"messages": [{"role": "user",
                        "content": user_text}]}
    ):
        if "messages" in chunk:
            for msg in chunk["messages"]:
                if hasattr(msg, "content") and msg.content:
                    yield msg.content

## 8.6 TTS Steps -- Cartesia Streaming Speech Synthesis

> **This cell requires a Cartesia API key.** Reference code to understand the concept.

Cartesia provides WebSocket-based low-latency TTS. It takes a text stream from an agent and generates audio chunks for each partial text on the fly.

### How Cartesia TTS works

1. **WebSocket connection**: Maintain a persistent connection by opening a WebSocket session with the `AsyncCartesia` client.
2. **Send text chunks**: Send text in units of tokens/sentences generated by the agent.
3. **Receive Audio Chunk**: Receive PCM audio bytes immediately for each text chunk
4. **Client Delivery**: Stream the received audio bytes to the client via WebSocket.

Because it is based on WebSocket, two-way real-time communication is possible without HTTP request/response overhead. The `sonic-2` model provides both natural speech and low latency.

```python
import cartesia

cartesia_client = cartesia.AsyncCartesia(
    api_key="your-cartesia-key"
)

async def text_to_speech_stream(text_stream):
    ws = await cartesia_client.tts.websocket()
    async for text_chunk in text_stream:
        audio_chunks = ws.send(
            model_id="sonic-2",
            transcript=text_chunk,
            voice_id="your-voice-id",
            stream=True,
            output_format={
                "container": "raw",
                "encoding": "pcm_s16le",
                "sample_rate": 24000,
            },
        )
        async for audio in audio_chunks:
            yield audio["audio"]
    await ws.close()
```

## 8.7 Pipeline Combination -- RunnableGenerator

LangChain's `RunnableGenerator` allows you to integrate asynchronous generators into your **Runnable pipeline**. This allows the entire data flow of STT output -> Agent processing -> TTS input to be organized into one pipeline.

### What is RunnableGenerator?

`RunnableGenerator` wraps an asynchronous generator function into LangChain's `Runnable` interface. If you do this:

- Chaining with other Runnables is possible using the `|` (pipe) operator.
- Standard methods such as LangChain’s `batch()` and `stream()` can be used.
- Suitable for patterns that receive an input stream and generate a converted output stream.

```python
from langchain_core.runnables import RunnableGenerator

async def transform_input(input_stream):
    async for text in input_stream:
        async for token in stream_agent_response(text):
            yield token

agent_runnable = RunnableGenerator(transform_input)
```

### 전체 파이프라인 연결 (개념 코드)

`asyncio.Queue`를 사용하여 STT의 최종 전사 결과를 에이전트로 전달하고, 에이전트의 스트리밍 응답을 TTS로 중계합니다. `Queue`는 비동기 Producer-Consumer 패턴의 핵심 구성 요소입니다.

```python
async def voice_pipeline(audio_input_stream):
    transcript_queue = asyncio.Queue()

    def on_final(transcript):
        if isinstance(transcript, aai.RealtimeFinalTranscript):
            transcript_queue.put_nowait(transcript.text)

    transcriber = aai.RealtimeTranscriber(
        sample_rate=16000, on_data=on_final
    )
    transcriber.connect()

    async for audio_chunk in audio_input_stream:
        transcriber.stream(audio_chunk)
        if not transcript_queue.empty():
            user_text = await transcript_queue.get()
            text_stream = stream_agent_response(user_text)
            async for audio in text_to_speech_stream(text_stream):
                yield audio
```

## 8.8 WebSocket Server -- Real-time two-way communication

> **Running the code in this cell will start the server.** This is reference code to help you understand the concept.

Handles two-way audio streaming with clients via WebSockets. Receiving and sending are **executed** simultaneously with `asyncio.gather`.

### Why use WebSockets

Voice agents require two-way, real-time communication:

- **Receive**: Client continuously transmits microphone audio to the server
- **Send**: The server continuously transmits synthesized voice audio to the client.

HTTP's request-response model is not suitable for this kind of two-way streaming. WebSockets support two-way data flow over a single TCP connection, and with `asyncio.gather`, ingress and egress run concurrently without blocking each other.

```python
import websockets
import asyncio

async def handle_client(websocket):
    transcriber = create_transcriber()
    transcriber.connect()

    async def receive_audio():
        async for message in websocket:
            if isinstance(message, bytes):
                transcriber.stream(message)

    async def send_audio():
        async for transcript in transcription_queue:
            text_stream = stream_agent_response(transcript)
            async for audio in text_to_speech_stream(text_stream):
                await websocket.send(audio)

    await asyncio.gather(receive_audio(), send_audio())

async def main():
    async with websockets.serve(handle_client, "0.0.0.0", 8765):
        print("Voice agent server on ws://0.0.0.0:8765")
        await asyncio.Future()
```

## 8.9 Performance Target -- Sub-700ms Latency

A key performance metric for voice agents is Time to First Audio (TTFA). For a natural conversation experience, this metric should be less than 700ms.

### Latency analysis

| steps | target time | Description |
|------|-----------|------|
| STT final transcription | ~200ms | Detect end of utterance -> final text (AssemblyAI built-in endpointing) |
| Agent first token | ~300ms | Enter text -> Create first response token (TTFT: Time to First Token) |
| TTS first audio | ~150ms | First text token -> first audio chunk (WebSocket based) |
| **Total** | **<700ms** | **End of speech -> First audio output** |

### Optimization techniques

| technique | effect | Implementation method |
|------|------|----------|
| Streaming STT | Eliminate full transcription wait | `RealtimeTranscriber` + partial result |
| Streaming Agent | Start TTS before completing all responses | Use `astream()` — instead of `ainvoke()` |
| Streaming TTS | Reduce time to first audio | WebSocket-based synthesis |
| Connection Pooling | Eliminate connection setup delays | WebSocket reuse (avoid creating new connection per request) |
| VAD | Prevent processing of silent sections | AssemblyAI built-in endpointing |
| response caching | Immediate response to frequently asked questions | Frequently Asked Questions Caching |

> **Tip**: Agent's TTFT (Time to First Token) accounts for the largest proportion of overall latency. Short system prompts and lightweight model selection are key to latency optimization.

## 8.10 Add tool to agent

The strength of the voice agent is that its Sandwich architecture allows it to leverage text-based tool. You can add a variety of tool features such as search, calendar management, weather lookup, and more.

This is the biggest difference from the S2S (Speech-to-Speech) method. The S2S model processes audio directly, making text-based tool calling difficult, but the Sandwich architecture allows agents to operate in text areas, so all existing LangChain tool can be used as is.

### tool Design Tips

tool of voice agents **fast response** is important. tool As the execution time increases, the latency of the entire pipeline increases:

- Designed mainly for light API calls
- Timeout setting required
- Apply caching if possible

In [ ]:
def weather_tool(city: str) -> str:
    """View the current weather in your city."""
    return f"{city} 날씨: 15도, 구름 조금"

def reminder_tool(time: str, message: str) -> str:
    """Set reminders at specified times."""
    return f"{time}에 알림 설정됨: {message}"

In [ ]:
voice_agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, calendar_tool,
           weather_tool, reminder_tool],
    system_prompt=(
        "You are a voice assistant."
        "Please keep your response concise and conversational in 1-2 sentences."
    ),
)

In [ ]:
# Agent standalone test (check operation with text input)
response = voice_agent.invoke(
    {"messages": [{"role": "user",
      "content": "How is the weather in Seoul?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)

## Summary

| Topic | Key Takeaways |
|------|----------|
| **Sandwich Architecture** | STT -> Agent -> TTS, each layer can be replaced independently, supports tool calling |
| **STT (AssemblyAI)** | `RealtimeTranscriber`, Producer-Consumer pattern, WebSocket streaming |
| **Agent (LangChain)** | `create_agent` + `astream()`, token-level streaming, tool integration |
| **TTS (Cartesia)** | WebSocket-based low-latency synthesis, instant audio conversion of partial text |
| **Pipeline Combination** | `RunnableGenerator`, asynchronous generator chaining |
| **Performance Goals** | Sub-700ms (STT ~200ms + Agent ~300ms + TTS ~150ms) |
| **tool Expansion** | Text-based tool can be applied to voice agents as is |

### Next Steps
→ **[09_production.ipynb](./09_production.ipynb)**: Learn production deployment.